<a href="https://colab.research.google.com/github/sruthi-analyst/sruthi-codeboosters-2026/blob/main/Day9/Day_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##AI Agent is an LLM that can:

- Use tools
- Take multiple steps
- Decide which action to take
- Act on the real world (does not need a human to perform the steps)
- Can access live data (not just trained data)
Can query database

AGENT: RAG
ChatBot: FAQ Bot

Chat bot - only returns a generated content explaining the process or topic that we asked.

##MUlti-step Reasoning: The ReAct Pattern
1. Think - understand user's intent
2. Plan - prepares pre requisutes and steps
3. Act - performs that steps
4. Respond - return the response

##Major methods in Agentic AI
1. get_*curr_structure_of_data-schema* ()
2. generate_*step-sql* () by calling LLM
3. execute_*steps-sql* () and returns results in a DataFrame

#AGENTIC AI PREREQUISITES

In [6]:
!pip install groq -q

print("Libraries installed successfully!")

Libraries installed successfully!


In [8]:
#Libraries required
import sqlite3          #dataase communication
import os               #store api key in environ vars safely
import pandas as pd     #work on the data set easily using a dataframe
from groq import Groq   #to access LLM models
import re               #to get required format

print("All libraries imported successfully")

All libraries imported successfully


In [9]:
import os
from google.colab import userdata

# Retrieve the secret securely
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API Key successfully loaded from Colab Secrets!")
except userdata.SecretNotFoundError:
    print("Error: Please add 'GROQ_API_KEY' to your Colab Secrets sidebar.")

API Key successfully loaded from Colab Secrets!


In [13]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client initialized successfully!")

#insert model
MODEL = "llama-3.1-8b-instant"

print("groq client initialised")
print(f"Using model: {MODEL}")

Groq client initialized successfully!
groq client initialised
Using model: llama-3.1-8b-instant


In [17]:
import io  #io = input output
csv_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/student_performance.csv")
print("Imported Dataset into DataFrame successfully!")

print("First 5 rows of the dataset")
print(csv_data.head())
print("\n Shape of dataset: ", csv_data.shape)
print("\n", csv_data.describe())

Imported Dataset into DataFrame successfully!
First 5 rows of the dataset
   student_id          name  age  gender        department  semester  \
0        1001  Aarav Sharma   19    Male  Computer Science         2   
1        1002   Priya Patel   20  Female  Computer Science         2   
2        1003   Rohit Verma   19    Male       Electronics         2   
3        1004   Sneha Reddy   20  Female        Mechanical         2   
4        1005    Arjun Nair   19    Male  Computer Science         2   

   math_score  science_score  english_score  programming_score  \
0          85             78             72                 91   
1          76             82             88                 79   
2          65             74             61                 55   
3          70             80             75                 48   
4          92             88             81                 95   

   attendance_percentage       city  admission_year  
0                     92     Mumbai       

In [21]:
conn = sqlite3.connect("study_statistics.db")
csv_data.to_sql("students", conn, if_exists="replace", index=False)
print("SQLite database loaded with Dataset successfully!")

#test db connection
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\nVerification {test_df['total_rows'][0]} rows in database")

SQLite database loaded with Dataset successfully!

Verification 30 rows in database


#AGENITC AI
## GET  ~*~   GENERATE  ~*~   EXECUTE

In [29]:
def get_schema(conn, table_name="students"):
  """
  Reads structure of database...
  PARAMS:
    conn : DB connection through ehich we are accessing the table
    table_name : name of table we need to access now
  ...
  """

  #Query SQLite's internal table info
  cursor = conn.cursor()  #connection to SQLite used to execute SQL query

  cursor.execute(f"PRAGMA table_info({table_name})")  #PRAGMA table_info is built-in python function to show all columns names of the table
  #PRAGMA table_info - a special SQLite command that return columns names
  columns = cursor.fetchall()
  #returns a list of tuples, one tuple per column
  #Example tuple : (0, 'student_id', 'INTEGER', 0, None, ...)

  #build a human-readable schema description
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns: ")

  for col in columns:
    #col[1]: column name(second element of the tuple)
    #col[2]: data type(third element)
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  #Add sample values to  help AI understand the data
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_data = cursor.fetchall()
  schema_lines.append("\nSample rows (first 3): ")

  for row in sample_data:
    schema_lines.append(f"  {row}")

  return "\n".join(schema_lines)
  #"\n".join : Joins all lines with a new line
schema = get_schema(conn)
print(schema)

Table: students
Columns: 
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - department (TEXT)
  - semester (INTEGER)
  - math_score (INTEGER)
  - science_score (INTEGER)
  - english_score (INTEGER)
  - programming_score (INTEGER)
  - attendance_percentage (INTEGER)
  - city (TEXT)
  - admission_year (INTEGER)

Sample rows (first 3): 
  (1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
  (1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
  (1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)


In [33]:
def generate_sql(user_question, schema_text, client, model):
  """
  Sends the user's questions and database schema to the Groq model.
  Groq LLM Generates a SQL query that answers the user's question.
  PARAMS:
    user_question: Qn typed by the user
    schema_text: ..
  RETURNS:
    A single string containing the generated SQL query.
  """

  #Define the system prompt = instruction we give
  system_prompt = f"""You are the developer of SQL.
  You are connected to a SQLite database with the following structure:
  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text — only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  #call groq api
  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature = 0.0
  )

  sql_query = response.choices[0].message.content.strip()
  return sql_query


In [35]:
question = "Show me all female students"
print(f"Question: {question}")
print("\nGenerating SQL...")

sql = generate_sql(question, schema, client, MODEL)
print(f"\nGenerated SQL:\n{sql}")


Question: Show me all female students

Generating SQL...

Generated SQL:
SELECT * FROM students WHERE gender = 'Female'


In [36]:
question = "List all students scored >90 in maths and programming"
print(f"Question: {question}")
print("\nGenerating SQL...")

sql = generate_sql(question, schema, client, MODEL)
print(f"\nGenerated SQL:\n{sql}")

Question: List all students scored >90 in maths and programming

Generating SQL...

Generated SQL:
SELECT * FROM students WHERE math_score > 90 AND programming_score > 90


In [37]:
question = "Create a new table with examination scores alone and add a column named total_score"
print(f"Question: {question}")
print("\nGenerating SQL...")

sql = generate_sql(question, schema, client, MODEL)
print(f"\nGenerated SQL:\n{sql}")

Question: Create a new table with examination scores alone and add a column named total_score

Generating SQL...

Generated SQL:
CREATE TABLE examination_scores AS
SELECT name, math_score, science_score, english_score, programming_score,
       math_score + science_score + english_score + programming_score AS total_score
FROM students;
